# 03 — Bağımsız Değişkenler: NDVI + Albedo + Geçirimsiz Yüzey

Üç değişken üretilir ve `02`'deki `grid_30m_lst.gpkg`'e kolon olarak eklenir:

| Değişken | Kaynak | Doğal çöz. | Yöntem |
|---|---|---|---|
| **NDVI** | Sentinel-2 SR Harmonized | 10 m | Yaz medyan, SCL bulut maskesi |
| **Albedo** | Sentinel-2 SR Harmonized | 10 m | Liang 2001 broadband formülü |
| **Impervious oranı** | ESA WorldCover v200 (2021) | 10 m | `class == 50` (built-up) → mean |

**Önkoşul:** `02_lst.ipynb` çalıştırılmış, `data/processed/grid_30m_lst.gpkg` mevcut.

**Çıktı:** `data/processed/grid_30m_variables.gpkg` — 30m grid + 4 değişken kolonu.

In [1]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    DATA_PROCESSED, FIGURES, TABLES,
    CRS_PROJECTED, GEE_PROJECT,
    S2_YEARS, S2_MONTHS, S2_CLOUD_THRESHOLD,
    WORLDCOVER_VERSION,
    LST_GRID_30M, NDVI_RASTER, ALBEDO_RASTER, IMPERVIOUS_RASTER,
    GRID_30M_VARIABLES,
)
from src.gee_utils import (
    init_ee, boundary_to_ee_geometry,
    s2_summer_median, esa_worldcover_impervious,
)
from src.variables import zonal_stats_to_grid

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

In [2]:
# --- Earth Engine init (auth zaten yapıldı) ---
import ee
project_id = GEE_PROJECT or os.environ.get("GEE_PROJECT")
if not project_id:
    raise RuntimeError("GEE_PROJECT yok. Notebook 02'deki talimatları izleyin.")
init_ee(project_id)
print(f"EE initialized — project: {project_id}")

EE initialized — project: asliutpm


In [3]:
# --- LST grid'ini ve pilot sınırı yükle ---
if not LST_GRID_30M.exists():
    raise FileNotFoundError(f"{LST_GRID_30M} yok. Önce 02_lst.ipynb çalıştırın.")

grid = gpd.read_file(LST_GRID_30M)
print(f"30m grid: {len(grid):,} hücre, kolonlar: {list(grid.columns)}")

boundary = gpd.read_file(DATA_PROCESSED.parent / "grid" / "pilot_boundary.gpkg")
region_ee = boundary_to_ee_geometry(boundary)

30m grid: 28,247 hücre, kolonlar: ['cell_id', 'i', 'j', 'parent_id', 'lst_mean', 'lst_count', 'lst_std', 'lst_median', 'geometry']


In [4]:
# --- Sentinel-2 yaz medyan kompoziti (NDVI + Albedo aynı pipeline) ---
s2_image = s2_summer_median(
    region=region_ee,
    years=S2_YEARS,
    months=S2_MONTHS,
    cloud_cover_max=S2_CLOUD_THRESHOLD,
)
print(f"Bantlar: {s2_image.bandNames().getInfo()}")

Bantlar: ['NDVI', 'ALBEDO']


In [5]:
# --- NDVI'yi GeoTIFF olarak indir (10 m doğal çöz.) ---
import geemap

if NDVI_RASTER.exists():
    print(f"Atlandı (mevcut): {NDVI_RASTER.name}")
else:
    print(f"İndiriliyor → {NDVI_RASTER.name}")
    geemap.ee_export_image(
        ee_object=s2_image.select("NDVI"),
        filename=str(NDVI_RASTER),
        scale=10,
        region=region_ee,
        file_per_band=False,
        crs="EPSG:32636",
    )
    print(f"  {NDVI_RASTER.stat().st_size/1024:.1f} KB")

İndiriliyor → ndvi_summer_median_2020_2024.tif
Generating URL ...
Please wait ...
Data downloaded to C:\Users\ercan\Desktop\AslıUTPM\data\processed\ndvi_summer_median_2020_2024.tif
  4073.4 KB


In [6]:
# --- Albedo'yu indir ---
if ALBEDO_RASTER.exists():
    print(f"Atlandı (mevcut): {ALBEDO_RASTER.name}")
else:
    print(f"İndiriliyor → {ALBEDO_RASTER.name}")
    geemap.ee_export_image(
        ee_object=s2_image.select("ALBEDO"),
        filename=str(ALBEDO_RASTER),
        scale=10,
        region=region_ee,
        file_per_band=False,
        crs="EPSG:32636",
    )
    print(f"  {ALBEDO_RASTER.stat().st_size/1024:.1f} KB")

İndiriliyor → albedo_summer_median_2020_2024.tif
Generating URL ...
Please wait ...
Data downloaded to C:\Users\ercan\Desktop\AslıUTPM\data\processed\albedo_summer_median_2020_2024.tif
  8264.0 KB


In [7]:
# --- ESA WorldCover impervious binary maskesi ---
if IMPERVIOUS_RASTER.exists():
    print(f"Atlandı (mevcut): {IMPERVIOUS_RASTER.name}")
else:
    impervious_image = esa_worldcover_impervious(region_ee, version=WORLDCOVER_VERSION)
    print(f"İndiriliyor → {IMPERVIOUS_RASTER.name}")
    geemap.ee_export_image(
        ee_object=impervious_image,
        filename=str(IMPERVIOUS_RASTER),
        scale=10,
        region=region_ee,
        file_per_band=False,
        crs="EPSG:32636",
    )
    print(f"  {IMPERVIOUS_RASTER.stat().st_size/1024:.1f} KB")

İndiriliyor → impervious_esa_worldcover_2021.tif
Generating URL ...
Please wait ...
Data downloaded to C:\Users\ercan\Desktop\AslıUTPM\data\processed\impervious_esa_worldcover_2021.tif
  25.1 KB


In [8]:
# --- 30m grid'e zonal stats ekle ---
print("NDVI zonal mean...")
grid = zonal_stats_to_grid(grid, NDVI_RASTER, stats=("mean",), prefix="ndvi_")

print("Albedo zonal mean...")
grid = zonal_stats_to_grid(grid, ALBEDO_RASTER, stats=("mean",), prefix="albedo_")

print("Impervious oranı (mean of binary)...")
grid = zonal_stats_to_grid(grid, IMPERVIOUS_RASTER, stats=("mean",), prefix="impervious_")

# 0-1 oran -> 0-100 yüzde
grid["impervious_pct"] = (grid["impervious_mean"] * 100).round(2)

print("\nKolonlar şimdi:")
for c in grid.columns:
    if c != "geometry":
        print(f"  {c}")

NDVI zonal mean...


C:\Users\ercan\anaconda3\envs\utpm\Lib\site-packages\rasterstats\io.py:335: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(


Albedo zonal mean...
Impervious oranı (mean of binary)...

Kolonlar şimdi:
  cell_id
  i
  j
  parent_id
  lst_mean
  lst_count
  lst_std
  lst_median
  ndvi_mean
  albedo_mean
  impervious_mean
  impervious_pct


In [ ]:
# --- Hızlı dağılım özeti ---
summary_stats = grid[["lst_mean", "ndvi_mean", "albedo_mean", "impervious_pct"]].describe().round(3)
summary_stats

In [ ]:
# --- Korelasyon (LST vs diğer 3) ---
corr = grid[["lst_mean", "ndvi_mean", "albedo_mean", "impervious_pct"]].corr().round(3)
print("Pearson korelasyon:")
print(corr)

print("\nBeklentiler:")
print("  LST vs NDVI:        negatif (yeşil = serin)")
print("  LST vs Albedo:      negatif/zayıf (yansıtıcı yüzey daha az ısınır, ama UHI etkisi)")
print("  LST vs Impervious:  pozitif (asfalt/beton = sıcak)")

In [ ]:
# --- 4 değişkenin haritası ---
fig, axes = plt.subplots(2, 2, figsize=(15, 13))
specs = [
    ("lst_mean",      "LST ortalama (°C)",        "inferno"),
    ("ndvi_mean",     "NDVI ortalama",            "YlGn"),
    ("albedo_mean",   "Albedo ortalama",          "Greys"),
    ("impervious_pct","Geçirimsiz oran (%)",      "OrRd"),
]
for ax, (col, title, cmap) in zip(axes.ravel(), specs):
    vals = grid[col].dropna()
    grid.plot(
        column=col, cmap=cmap, ax=ax,
        legend=True, legend_kwds={"label": title, "shrink": 0.6},
        linewidth=0,
        vmin=np.percentile(vals, 2), vmax=np.percentile(vals, 98),
    )
    boundary.to_crs(CRS_PROJECTED).boundary.plot(
        ax=ax, color="#0a9396", linewidth=0.8,
    )
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.ticklabel_format(style="plain")
plt.tight_layout()
plt.savefig(FIGURES / "03_variables_maps.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# --- Korelasyon ısı haritası + scatter (LST vs her biri) ---
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            ax=axes[0], cbar_kws={"label": "Pearson r"}, vmin=-1, vmax=1)
axes[0].set_title("Korelasyon matrisi")

for ax, col, color in zip(
    axes[1:], ["ndvi_mean", "albedo_mean", "impervious_pct"],
    ["#2A9D8F", "#264653", "#E76F51"],
):
    sub = grid[["lst_mean", col]].dropna().sample(min(5000, len(grid)), random_state=42)
    ax.scatter(sub[col], sub["lst_mean"], s=2, alpha=0.3, color=color)
    ax.set_xlabel(col)
    ax.set_ylabel("lst_mean (°C)")
    r = sub.corr().iloc[0, 1]
    ax.set_title(f"LST vs {col}  (r={r:.2f})")

plt.tight_layout()
plt.savefig(FIGURES / "03_variables_corr.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# --- Kaydet + summary ---
grid.to_file(GRID_30M_VARIABLES, driver="GPKG")
print(f"Kaydedildi: {GRID_30M_VARIABLES} ({GRID_30M_VARIABLES.stat().st_size/1024/1024:.2f} MB)")

summary = pd.DataFrame({
    "degisken": ["LST", "NDVI", "Albedo", "Impervious_pct"],
    "kaynak":   ["Landsat 8/9 C2L2", "Sentinel-2 SR", "Sentinel-2 SR (Liang 2001)", f"ESA WorldCover {WORLDCOVER_VERSION}"],
    "birim":    ["°C", "-", "-", "%"],
    "min":      [grid[c].min() for c in ["lst_mean", "ndvi_mean", "albedo_mean", "impervious_pct"]],
    "max":      [grid[c].max() for c in ["lst_mean", "ndvi_mean", "albedo_mean", "impervious_pct"]],
    "medyan":   [grid[c].median() for c in ["lst_mean", "ndvi_mean", "albedo_mean", "impervious_pct"]],
    "r_with_LST":[1.0, corr.loc["ndvi_mean", "lst_mean"], corr.loc["albedo_mean", "lst_mean"], corr.loc["impervious_pct", "lst_mean"]],
}).round(3)
summary.to_csv(TABLES / "03_variables_summary.csv", index=False, encoding="utf-8-sig")
summary

## Özet

- **NDVI** ve **Albedo** Sentinel-2 SR Harmonized'dan yaz medyan (2020-2024 Hz-Ağ) kompoziti olarak üretildi; SCL bulut maskesi.
- **Albedo** Liang (2001) short-wave broadband formülü, Sentinel-2 5 bant kombinasyonu.
- **Geçirimsiz yüzey oranı** ESA WorldCover v200 (2021)'den `class == 50` binary mean.
- 30 m grid'e zonal mean ile bağlandı; LST ile korelasyonlar yorumlandı.

## Sonraki Adım

`04_building_height.ipynb` — İmar verisinden (`kat_adedi × 3.0 m`) bina yüksekliği + buffer cascade imputation (10/100/500/1000 m) + GHSL doğrulama. Yapı yoğunluğu ve yol yoğunluğu bu hafta da devreye girer.